In [7]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.wkb import loads

df = pd.read_csv('../Data/road_info_detail_withmonth.csv')

# Aggregate monthly observations per road segment
agg_dict = {
    'geom': 'first', 'maxspeed': 'first', 'lu_label': 'first',
    'total_build_count': 'first', 'imd_deci': 'first', 'cri_deci': 'first',
    'crime_count': ['mean', 'std', 'sum', 'count'],
    'violence_sexual': 'sum', 'anti_social': 'sum', 'drugs': 'sum',
    'other_theft': 'sum', 'robbery': 'sum', 'public_order': 'sum'
}

df_summary = df.groupby('road_id').agg(agg_dict)
df_summary.columns = ['_'.join(c).strip('_') for c in df_summary.columns]
df_summary = df_summary.reset_index()

# Rename fields
df_summary.rename(columns={
    'geom_first': 'geom', 'maxspeed_first': 'maxspeed',
    'lu_label_first': 'lu_label', 'total_build_count_first': 'build_count',
    'imd_deci_first': 'imd_deci', 'cri_deci_first': 'cri_deci',
    'crime_count_mean': 'pred_crime', 'crime_count_std': 'crime_std',
    'crime_count_sum': 'total_crime', 'crime_count_count': 'months',
    'violence_sexual_sum': 'violence', 'anti_social_sum': 'anti_social',
    'drugs_sum': 'drugs', 'other_theft_sum': 'theft',
    'robbery_sum': 'robbery', 'public_order_sum': 'public_order'
}, inplace=True)

# Calculate standard error and 95% confidence bounds
df_summary['crime_std'] = df_summary['crime_std'].fillna(0)
df_summary['se'] = df_summary['crime_std'] / np.sqrt(df_summary['months'])
df_summary['ci_margin'] = (1.96 * df_summary['se']).round(2)
df_summary['pred_crime'] = df_summary['pred_crime'].round(2)
df_summary['ci_upper'] = (df_summary['pred_crime'] + df_summary['ci_margin']).round(2)
df_summary['ci_lower'] = (df_summary['pred_crime'] - df_summary['ci_margin']).clip(lower=0).round(2)

# Convert geometry from EPSG:27700 to WGS84 EPSG:4326
df_summary['geometry'] = df_summary['geom'].apply(lambda x: loads(x, hex=True))
gdf = gpd.GeoDataFrame(df_summary.drop(columns=['geom']), geometry='geometry', crs="EPSG:27700").to_crs(epsg=4326)
gdf.fillna(0).to_file("../Data/crime_roads.geojson", driver="GeoJSON")

In [8]:
df_time_testing = pd.read_csv('../Data/time_testing_results.csv')
df_crime_history_train = pd.read_csv('../Data/df_crime_history_train.csv')
df_crime_history_test = pd.read_csv('../Data/df_crime_history_test.csv')

In [15]:
df_time_testing

,3215,3330,3386,3418,3423,3456,3545,3552,3627,3654,...,6222,6246,6249,6333,6303,6371,6412,6378,6416,6417
0,[0.34109524],[0.2506346],[0.1843592],[0.21139511],[0.16199347],[0.3519481],[0.23827027],[0.17726965],[0.16784497],[0.1754667],...,[0.46494702],[0.21779941],[0.5355271],[0.49175283],[0.21022244],[0.42114338],[0.23751162],[0.2858324],[0.22156318],[0.21710931]
1,[0.3268652],[0.2420047],[0.17068416],[0.19402009],[0.14318442],[0.33574468],[0.22205296],[0.16214034],[0.14806788],[0.15685962],...,[0.45149362],[0.20411737],[0.5395889],[0.47531238],[0.1937722],[0.41042763],[0.22304007],[0.2763644],[0.20575762],[0.2025874]
2,[0.37491083],[0.2507658],[0.18614917],[0.20330814],[0.1408789],[0.33467835],[0.26389974],[0.19189456],[0.17764316],[0.16042945],...,[0.5144553],[0.20053881],[0.5675111],[0.48231885],[0.21105777],[0.43128052],[0.27361736],[0.29758272],[0.2356273],[0.21151286]
3,[0.14885253],[0.08824721],[0.04905989],[0.05923667],[0.03574263],[0.10259365],[0.09286383],[0.05243541],[0.04232053],[0.03952465],...,[0.22941306],[0.06033421],[0.33387038],[0.20247003],[0.06543946],[0.17087379],[0.085678],[0.1120681],[0.077781],[0.06072465]
4,[0.25773072],[0.17812149],[0.12195819],[0.14386883],[0.10785108],[0.24268231],[0.16471799],[0.11962523],[0.10545252],[0.11620704],...,[0.3222701],[0.14934076],[0.4287703],[0.37036017],[0.14396398],[0.3300984],[0.1672718],[0.20282485],[0.15385967],[0.14630844]
5,[0.41172343],[0.30615354],[0.22975689],[0.27108514],[0.19917002],[0.45282274],[0.2917747],[0.2184205],[0.21561418],[0.20808789],...,[0.59755504],[0.26604772],[0.6543082],[0.60914814],[0.2565419],[0.50387466],[0.28503153],[0.35307154],[0.27143723],[0.26964694]
6,[0.02604972],[0.01318918],[0.00532483],[0.00484527],[0.00380323],[0.01901096],[0.00748662],[0.00455879],[0.00377191],[0.00486685],...,[0.03734042],[0.00819006],[0.06971139],[0.05101935],[0.00649066],[0.03629436],[0.00807032],[0.02079917],[0.01012525],[0.00558013]
7,[0.14885253],[0.08824721],[0.04905989],[0.05923667],[0.03574263],[0.10259365],[0.09286383],[0.05243541],[0.04232053],[0.03952465],...,[0.22941306],[0.06033421],[0.33387038],[0.20247003],[0.06543946],[0.17087379],[0.085678],[0.1120681],[0.077781],[0.06072465]
8,[0.48405084],[0.3383056],[0.24499074],[0.24735437],[0.1519883],[0.41021925],[0.34860843],[0.22830129],[0.21165511],[0.20503065],...,[0.6210582],[0.25193185],[0.7922515],[0.6908413],[0.25676826],[0.4662918],[0.34716898],[0.33019698],[0.2884819],[0.26072717]
9,[1.4011927],[1.3434303],[1.4041486],[1.2499174],[1.0349245],[1.1651958],[1.5362183],[1.3799311],[1.4039723],[1.2760696],...,[1.2928596],[1.1941257],[1.3390566],[1.3461083],[1.2916074],[1.0476815],[1.5203485],[1.1195285],[1.3528376],[1.2594419]
